In [0]:
from pyspark.sql.functions import (
    col, round, avg, max, min,
    stddev, count, desc, rank
)
from pyspark.sql.window import Window

# Load Silver Parquet data
df_silver = spark.read.parquet("/mnt/silver/stock_data/")

print(f"Silver data loaded: {df_silver.count():,} rows")
print(f"Columns: {df_silver.columns}")
df_silver.show(5)

Silver data loaded: 500 rows
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'daily_return_pct', 'price_change', 'moving_avg_7day', 'daily_range', 'is_positive_day', 'symbol']
+----------+-------+--------+--------+------+------------+----------------+------------+---------------+-----------+---------------+------+
|      date|   open|    high|     low| close|      volume|daily_return_pct|price_change|moving_avg_7day|daily_range|is_positive_day|symbol|
+----------+-------+--------+--------+------+------------+----------------+------------+---------------+-----------+---------------+------+
|2025-10-27| 439.98|  460.16|  438.69|452.42|1.05867547E8|            NULL|        NULL|         452.42|      21.47|             No|  TSLA|
|2025-10-28|454.775|   467.0|   451.6|460.55| 8.0185667E7|             1.8|        8.13|         456.49|       15.4|            Yes|  TSLA|
|2025-10-29|  462.5|   465.7|  452.65|461.51| 6.7983544E7|            0.21|        0.96|         458.16|      13

In [0]:
# Register as temp view so we can run SQL directly
df_silver.createOrReplaceTempView("stock_silver")

print("Temp view 'stock_silver' created!")
print("You can now run SQL queries on this view.")

Temp view 'stock_silver' created!
You can now run SQL queries on this view.


In [0]:
query1 = """
SELECT
    symbol,
    ROUND(AVG(daily_return_pct), 3) AS avg_daily_return_pct,
    ROUND(SUM(CASE WHEN is_positive_day = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS positive_days_pct,
    ROUND(MIN(close), 2) AS min_close,
    ROUND(MAX(close), 2) AS max_close,
    ROUND(MAX(close) - MIN(close), 2) AS price_range
FROM stock_silver
GROUP BY symbol
ORDER BY avg_daily_return_pct DESC
"""

df_best_stock = spark.sql(query1)
print("=== Best Performing Stocks ===")
df_best_stock.show()

=== Best Performing Stocks ===
+------+--------------------+-----------------+---------+---------+-----------+
|symbol|avg_daily_return_pct|positive_days_pct|min_close|max_close|price_range|
+------+--------------------+-----------------+---------+---------+-----------+
| GOOGL|               0.126|             48.0|   267.47|   343.69|      76.22|
|  AAPL|              -0.046|             48.0|    246.7|   286.19|      39.49|
|  AMZN|              -0.081|             52.0|   198.79|    254.0|      55.21|
|  TSLA|              -0.178|             49.0|   367.96|   489.88|     121.92|
|  MSFT|              -0.284|             44.0|   384.47|   542.07|      157.6|
+------+--------------------+-----------------+---------+---------+-----------+



In [0]:
query2 = """
SELECT
    symbol,
    ROUND(AVG(daily_range), 2) AS avg_daily_range,
    ROUND(stddev(daily_return_pct), 3) AS volatility_stddev,
    ROUND(MAX(daily_range), 2) AS max_single_day_range,
    COUNT(*) AS trading_days
FROM stock_silver
GROUP BY symbol
ORDER BY volatility_stddev DESC
"""

df_volatility = spark.sql(query2)
print("=== Stock Volatility Ranking ===")
df_volatility.show()

=== Stock Volatility Ranking ===
+------+---------------+-----------------+--------------------+------------+
|symbol|avg_daily_range|volatility_stddev|max_single_day_range|trading_days|
+------+---------------+-----------------+--------------------+------------+
|  TSLA|          14.68|            2.506|                34.2|         100|
|  AMZN|           5.35|            2.042|               11.13|         100|
|  MSFT|           8.95|            1.775|               21.48|         100|
| GOOGL|           7.95|            1.672|               26.23|         100|
|  AAPL|           5.26|            1.317|               15.54|         100|
+------+---------------+-----------------+--------------------+------------+



In [0]:
query3 = """
SELECT
    symbol,
    DATE_FORMAT(date, 'yyyy-MM') AS month,
    ROUND(AVG(close), 2) AS avg_monthly_close,
    ROUND(AVG(daily_return_pct), 3) AS avg_monthly_return,
    SUM(volume) AS total_monthly_volume
FROM stock_silver
GROUP BY symbol, DATE_FORMAT(date, 'yyyy-MM')
ORDER BY symbol, month DESC
LIMIT 30
"""

df_monthly = spark.sql(query3)
print("=== Monthly Return Comparison ===")
df_monthly.show(30)

=== Monthly Return Comparison ===
+------+-------+-----------------+------------------+--------------------+
|symbol|  month|avg_monthly_close|avg_monthly_return|total_monthly_volume|
+------+-------+-----------------+------------------+--------------------+
|  AAPL|2026-03|           257.29|            -0.419|        5.18788659E8|
|  AAPL|2026-02|           269.18|             0.118|        9.88325816E8|
|  AAPL|2026-01|           257.65|            -0.225|       1.036170325E9|
|  AAPL|2025-12|           276.31|            -0.114|        9.22283649E8|
|  AAPL|2025-11|           271.66|             0.167|        8.76481453E8|
|  AAPL|2025-10|           268.68|             0.572|        3.31817027E8|
|  AMZN|2026-03|           211.65|            -0.135|        6.85707019E8|
|  AMZN|2026-02|           212.05|            -0.663|       1.259918927E9|
|  AMZN|2026-01|           239.08|             0.196|        8.41068727E8|
|  AMZN|2025-12|           229.26|            -0.043|        7.874

In [0]:
query4 = """
SELECT
    symbol,
    date,
    close,
    daily_return_pct,
    price_change
FROM (
    SELECT *,
        RANK() OVER (PARTITION BY symbol ORDER BY daily_return_pct DESC) AS day_rank
    FROM stock_silver
    WHERE daily_return_pct IS NOT NULL
)
WHERE day_rank <= 5
ORDER BY symbol, day_rank
"""

df_best_days = spark.sql(query4)
print("=== Top 5 Best Days Per Stock ===")
df_best_days.show(25)

=== Top 5 Best Days Per Stock ===
+------+----------+------+----------------+------------+
|symbol|      date| close|daily_return_pct|price_change|
+------+----------+------+----------------+------------+
|  AAPL|2026-02-02|270.01|            4.06|       10.53|
|  AAPL|2026-02-17|263.88|            3.17|         8.1|
|  AAPL|2026-01-26|255.41|            2.97|        7.37|
|  AAPL|2026-02-04|276.49|             2.6|        7.01|
|  AAPL|2025-10-27|268.81|            2.28|        5.99|
|  AMZN|2025-10-31|244.22|            9.58|       21.36|
|  AMZN|2025-11-03| 254.0|             4.0|        9.78|
|  AMZN|2026-03-04|216.82|            3.88|        8.09|
|  AMZN|2026-01-06|240.93|            3.38|        7.87|
|  AMZN|2026-01-05|233.06|             2.9|        6.56|
| GOOGL|2025-11-24|318.58|            6.31|       18.92|
| GOOGL|2025-11-10| 290.1|            4.04|       11.27|
| GOOGL|2026-02-20|314.98|            4.01|       12.13|
| GOOGL|2025-11-21|299.66|            3.53|       10.2

In [0]:
query5 = """
SELECT
    symbol,
    date,
    close,
    moving_avg_7day,
    CASE
        WHEN close > moving_avg_7day THEN 'Above MA'
        WHEN close < moving_avg_7day THEN 'Below MA'
        ELSE 'At MA'
    END AS price_vs_ma
FROM stock_silver
ORDER BY symbol, date DESC
LIMIT 50
"""

df_ma_trend = spark.sql(query5)
print("=== 7-Day Moving Average Trend ===")
df_ma_trend.show(50)

=== 7-Day Moving Average Trend ===
+------+----------+------+---------------+-----------+
|symbol|      date| close|moving_avg_7day|price_vs_ma|
+------+----------+------+---------------+-----------+
|  AAPL|2026-03-19|248.96|         253.23|   Below MA|
|  AAPL|2026-03-18|249.94|         254.93|   Below MA|
|  AAPL|2026-03-17|254.23|         256.35|   Below MA|
|  AAPL|2026-03-16|252.82|         256.81|   Below MA|
|  AAPL|2026-03-13|250.12|         257.88|   Below MA|
|  AAPL|2026-03-12|255.76|         259.65|   Below MA|
|  AAPL|2026-03-11|260.81|         260.79|   Above MA|
|  AAPL|2026-03-10|260.83|         261.35|   Below MA|
|  AAPL|2026-03-09|259.88|         261.83|   Below MA|
|  AAPL|2026-03-06|257.46|          263.7|   Below MA|
|  AAPL|2026-03-05|260.29|         266.09|   Below MA|
|  AAPL|2026-03-04|262.52|         267.78|   Below MA|
|  AAPL|2026-03-03|263.75|         268.31|   Below MA|
|  AAPL|2026-03-02|264.72|         268.43|   Below MA|
|  AAPL|2026-02-27|264.18|    

In [0]:
# Save all 5 query results as Parquet to Gold layer
df_best_stock.write.mode("overwrite").parquet("/mnt/gold/best_performing_stocks/")
df_volatility.write.mode("overwrite").parquet("/mnt/gold/volatility_ranking/")
df_monthly.write.mode("overwrite").parquet("/mnt/gold/monthly_returns/")
df_best_days.write.mode("overwrite").parquet("/mnt/gold/best_trading_days/")
df_ma_trend.write.mode("overwrite").parquet("/mnt/gold/moving_avg_trend/")

print("All Gold layer outputs saved!")

# Verify
folders = dbutils.fs.ls("/mnt/gold/")
print("\nFolders in Gold container:")
for f in folders:
    print(f"  {f.name}")

All Gold layer outputs saved!

Folders in Gold container:
  best_performing_stocks/
  best_trading_days/
  monthly_returns/
  moving_avg_trend/
  volatility_ranking/


In [0]:
print("=" * 50)
print("GOLD LAYER — KEY FINDINGS SUMMARY")
print("=" * 50)

# Best performing stock
best = df_best_stock.first()
print(f"\nBest avg daily return : {best['symbol']} ({best['avg_daily_return_pct']}%)")

# Most volatile stock
most_volatile = df_volatility.first()
print(f"Most volatile stock   : {most_volatile['symbol']} (stddev: {most_volatile['volatility_stddev']})")

# Highest positive days
best_consistency = df_best_stock.orderBy(desc("positive_days_pct")).first()
print(f"Most consistent stock : {best_consistency['symbol']} ({best_consistency['positive_days_pct']}% positive days)")

print("\nGold layer complete!")
print("=" * 50)

GOLD LAYER — KEY FINDINGS SUMMARY

Best avg daily return : GOOGL (0.126%)
Most volatile stock   : TSLA (stddev: 2.506)
Most consistent stock : AMZN (52.0% positive days)

Gold layer complete!
